In [2]:
# from fastmcp import FastMCP

# # 初始化mcp
# mcp = FastMCP("Math")

# @mcp.tool()
# def add(x: float, y: float) -> float:
#   """两数相加"""
#   return x + y

# @mcp.tool()
# def multiply(x: float, y: float) -> float:
#   """两数相乘"""
#   return x * y

# # 启动mcp服务，通信方式设置为stdio
# if __name__ == "__main__":
#   mcp.run(transport="stdio")

from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
  "math": {
    "transport": "stdio",
    "command": "python",
    "args": ["mcp_custom.py"]
  }
})

tools = await client.get_tools()

print(tools)

[StructuredTool(name='add', description='两数相加', args_schema={'additionalProperties': False, 'properties': {'x': {'type': 'number'}, 'y': {'type': 'number'}}, 'required': ['x', 'y'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, handle_tool_error=<function _handle_mcp_tool_error at 0x111e271a0>, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x10f852840>), StructuredTool(name='multiply', description='两数相乘', args_schema={'additionalProperties': False, 'properties': {'x': {'type': 'number'}, 'y': {'type': 'number'}}, 'required': ['x', 'y'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, handle_tool_error=<function _handle_mcp_tool_error at 0x111e271a0>, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x111ed9a80>)]


In [6]:
from langchain.agents import create_agent
from dotenv import load_dotenv

load_dotenv()

agent = create_agent(
  model="deepseek-flash",
  tools=tools,
  system_prompt="你是一个数学助手，可以调用工具回答用户提问"
)

response = await agent.ainvoke(
  {"messages": "25 * 25等于多少？12 + 12等于多少？"}
)

for message in response["messages"]:
  message.pretty_print()

================================ Human Message =================================

25 * 25等于多少？12 + 12等于多少？
================================== Ai Message ==================================
Tool Calls:
  multiply (call_00_M3aSwT2LXln4Y9IWAz0T2893)
 Call ID: call_00_M3aSwT2LXln4Y9IWAz0T2893
  Args:
    x: 25
    y: 25
  add (call_01_nvPB818t3JbvLQhU502g7657)
 Call ID: call_01_nvPB818t3JbvLQhU502g7657
  Args:
    x: 12
    y: 12
================================= Tool Message =================================
Name: multiply

[{'type': 'text', 'text': '625.0', 'id': 'lc_2248163a-7055-435a-93d1-e3e07ed249cd'}]
================================= Tool Message =================================
Name: add

[{'type': 'text', 'text': '24.0', 'id': 'lc_818ca3fd-1a42-45b4-8673-7a700952ff5e'}]
================================== Ai Message ==================================

25 × 25 = **625**  
12 + 12 = **24**
